# Make your own claim

Write a tiny results file and a one-claim Markdown note, check it, then change
the number and watch the verdict turn `contradicted`. Nothing here needs the
repo's examples or a model server; it writes everything to a temporary folder
and a temporary ledger.

Copy this notebook before editing it: the copy that ships with the skill is
overwritten whenever attestation updates it.

In [1]:
import json
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# Everything this notebook writes goes to a throwaway folder and a throwaway
# ledger. Your real database and workspace are never read or changed.
WORK = Path(tempfile.mkdtemp(prefix="attest-demo-"))
os.environ["ATTEST_DB"] = str(WORK / "demo.db")
HOME = WORK / "hermes"
os.environ["HERMES_HOME"] = str(HOME)
# Point these at the temp folder rather than just unsetting them: every `attest`
# call reloads a checkout's .env, which would put an unset variable back.
os.environ["LEDGER_METRIC_DIRECTION_FILE"] = str(HOME / "metric_direction.toml")
os.environ["LEDGER_CORPUS_FILE"] = str(HOME / "corpus.toml")
os.environ["RESEARCH_ROOT"] = str(WORK / "workspace")


def _find_attest():
    found = shutil.which("attest", path=str(Path(sys.executable).parent)) or shutil.which("attest")
    if found is None:
        raise RuntimeError(
            "`attest` is not on PATH: run this notebook in the attestation environment"
        )
    return found


_ATTEST = _find_attest()


def attest(command, cwd):
    """Run `attest <command>` offline in `cwd`, print what it prints, return its exit code."""
    print("$ attest " + command)
    done = subprocess.run(
        [_ATTEST, *shlex.split(command)], cwd=cwd, capture_output=True, text=True, timeout=60
    )
    text = (done.stdout + done.stderr).rstrip()
    print(text.replace(str(WORK), "<tmp>").replace(str(Path.home()), "~"))
    print(f"[exit {done.returncode}]")
    return done.returncode


print("Working in a temporary folder with a temporary ledger.")

Working in a temporary folder with a temporary ledger.


## 1. A run, and a sentence that cites it

The run is a JSON file under `results/`. The ledger reads files like this where
they already are; nothing is registered in advance. The claim names the run
(`project/run`), the metric and the value, and sits in a comment beside the
prose so the note renders unchanged.

In [2]:
OWN = WORK / "own"
(OWN / "mywork" / "results").mkdir(parents=True)
(OWN / "mywork" / "results" / "run_a.json").write_text(
    json.dumps({"tag": "run_a", "corpus": "demo", "accuracy": 0.912, "n_records": 500})
)
note = OWN / "mywork" / "notes.md"
note.write_text(
    "Run A reaches an accuracy of 0.912.\n"
    "<!-- claim: mywork/run_a metric=accuracy value=0.912 tol=0.001 -->\n"
)
print(note.read_text())

Run A reaches an accuracy of 0.912.
<!-- claim: mywork/run_a metric=accuracy value=0.912 tol=0.001 -->



## 2. Scan, then check

In [3]:
attest("runs scan --root .", cwd=OWN)
attest("claims mywork/notes.md", cwd=OWN)

$ attest runs scan --root .


  mywork                       1 run(s)
1 run(s) across 1 project(s)
[exit 0]
$ attest claims mywork/notes.md


  supported     notes.md:2                   accuracy=0.912  accuracy=0.912 in run_a

1 claim(s): 1 supported
[exit 0]


0

## 3. Change the number

Suppose the note is edited to say 0.931 and nobody re-ran anything. The run on
disk still records 0.912.

In [4]:
note.write_text(note.read_text().replace("0.912 tol", "0.931 tol").replace("of 0.912", "of 0.931"))
print(note.read_text())
attest("claims mywork/notes.md", cwd=OWN)

Run A reaches an accuracy of 0.931.
<!-- claim: mywork/run_a metric=accuracy value=0.931 tol=0.001 -->

$ attest claims mywork/notes.md


  contradicted  notes.md:2                   accuracy=0.931  document says 0.931, run records 0.912 (tolerance 0.001)

1 claim(s): 1 contradicted
[exit 1]


1

`contradicted`, and the message names both numbers: the note says 0.931, the
run records 0.912. Either the note or the run is wrong, and the check cannot
tell you which; that is yours to decide. Exit code 1 means the check worked.

## 4. A claim with nothing behind it

Point the claim at a run that does not exist and the verdict is different:
`unsupported` is "nothing here backs this", not "false".

In [5]:
note.write_text(note.read_text().replace("mywork/run_a", "mywork/run_b").replace("0.931", "0.912"))
attest("claims mywork/notes.md", cwd=OWN)

$ attest claims mywork/notes.md


  unsupported   notes.md:2                   accuracy=0.912  no run mywork/run_b in the ledger -- the claim may be true, but nothing here backs it

1 claim(s): 1 unsupported
[exit 0]


0